# XGBoost Ablation Study — IEEE-CIS Fraud Detection

This notebook runs two controlled ablation experiments:

1. **Without engineered features:** removes `Amount_Relative_To_Product`, `P_emaildomain_Frequency`, and `R_emaildomain_Frequency`.
2. **Without identity features:** removes columns beginning with `id_`.

It retrains all three conditions (full-feature baseline and two ablations) using the same chronological 70/10/20 split, XGBoost configuration, preprocessing approach, validation threshold selection, and untouched test set.

**Important:** This notebook generates actual results when executed in Google Colab. Do not report numerical ablation findings until the cells finish successfully. The original baseline metrics in the project report are shown only as historical reference; the controlled baseline below is retrained in this notebook.


## 1. Mount Google Drive and load the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, warnings
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Fraud_Detection_Project"
RESULTS_DIR = os.path.join(PROJECT_DIR, "Model_Results", "Ablation_Study")
os.makedirs(RESULTS_DIR, exist_ok=True)

parquet_path = os.path.join(PROJECT_DIR, "merged_dataset.parquet")

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
else:
    transaction_path = os.path.join(PROJECT_DIR, "train_transaction.csv")
    identity_path = os.path.join(PROJECT_DIR, "train_identity.csv")
    transaction = pd.read_csv(transaction_path, low_memory=False)
    identity = pd.read_csv(identity_path, low_memory=False)
    df = transaction.merge(identity, on="TransactionID", how="left")
    del transaction, identity
    gc.collect()

df = df.sort_values("TransactionDT").reset_index(drop=True)
print("Merged dataset shape:", df.shape)
print(df["isFraud"].value_counts(dropna=False))


## 2. Recreate the chronological split

In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.80)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train:", train_df.shape, "Fraud:", int(train_df.isFraud.sum()))
print("Validation:", val_df.shape, "Fraud:", int(val_df.isFraud.sum()))
print("Test:", test_df.shape, "Fraud:", int(test_df.isFraud.sum()))


## 3. Define preprocessing and feature engineering

All learned statistics (product medians, frequency counts, and categorical mappings) are derived from the training partition only. `TransactionID` and `isFraud` are excluded from predictors. Missing numeric values are left for XGBoost's native missing-value handling. Categorical values are mapped to integer codes, with unseen categories assigned `-1`.


In [ ]:
TARGET = "isFraud"
ENGINEERED = [
    "Amount_Relative_To_Product",
    "P_emaildomain_Frequency",
    "R_emaildomain_Frequency"
]

def fit_transform_features(train_part, other_parts, remove_engineered=False, remove_identity=False):
    # Start from independent copies; never modify the source split frames.
    tr = train_part.copy()
    others = [part.copy() for part in other_parts]

    # Training-only engineered-feature statistics.
    if "ProductCD" in tr.columns and "TransactionAmt" in tr.columns:
        product_medians = tr.groupby("ProductCD")["TransactionAmt"].median()
        global_median = tr["TransactionAmt"].median()

        def add_relative_amount(frame):
            med = frame["ProductCD"].map(product_medians)
            frame["Amount_Relative_To_Product"] = (
                frame["TransactionAmt"] / med.fillna(global_median)
            )

        add_relative_amount(tr)
        for frame in others:
            add_relative_amount(frame)

    for col in ["P_emaildomain", "R_emaildomain"]:
        if col in tr.columns:
            counts = tr[col].value_counts(dropna=False)
            feature_name = f"{col}_Frequency"
            tr[feature_name] = tr[col].map(counts).fillna(0)
            for frame in others:
                frame[feature_name] = frame[col].map(counts).fillna(0)

    # Exclude target and identifier.
    drop_cols = [c for c in ["TransactionID", TARGET] if c in tr.columns]
    X_train = tr.drop(columns=drop_cols)
    X_others = [f.drop(columns=drop_cols) for f in others]

    # Ablation switches.
    if remove_engineered:
        cols = [c for c in ENGINEERED if c in X_train.columns]
        X_train = X_train.drop(columns=cols)
        X_others = [x.drop(columns=cols) for x in X_others]

    if remove_identity:
        cols = [c for c in X_train.columns if c.startswith("id_")]
        X_train = X_train.drop(columns=cols)
        X_others = [x.drop(columns=cols) for x in X_others]

    # Convert object/category columns to consistent train-fitted integer codes.
    categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in categorical_cols:
        train_values = X_train[col].astype("string").fillna("__MISSING__")
        categories = pd.Index(train_values.unique())
        mapping = {value: i for i, value in enumerate(categories)}

        X_train[col] = train_values.map(mapping).astype("int32")
        for X in X_others:
            values = X[col].astype("string").fillna("__MISSING__")
            X[col] = values.map(mapping).fillna(-1).astype("int32")

    # Ensure numeric-compatible columns and identical order.
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_others = [X.reindex(columns=X_train.columns).replace([np.inf, -np.inf], np.nan)
                for X in X_others]

    return X_train, X_others


## 4. Train baseline and ablation models

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    roc_auc_score, confusion_matrix
)

def best_f1_threshold(y_true, probabilities):
    thresholds = np.linspace(0.01, 0.99, 300)
    scores = [f1_score(y_true, probabilities >= t, zero_division=0) for t in thresholds]
    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx]), float(scores[best_idx])

def run_experiment(name, remove_engineered=False, remove_identity=False):
    X_train, (X_val, X_test) = fit_transform_features(
        train_df, [val_df, test_df],
        remove_engineered=remove_engineered,
        remove_identity=remove_identity
    )

    y_train = train_df[TARGET].astype(int)
    y_val = val_df[TARGET].astype(int)
    y_test = test_df[TARGET].astype(int)

    model = XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=27.43,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        max_bin=256,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    val_prob = model.predict_proba(X_val)[:, 1]
    threshold, val_f1 = best_f1_threshold(y_val, val_prob)

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()

    result = {
        "Experiment": name,
        "Features": X_train.shape[1],
        "Best_iteration": int(model.best_iteration) if model.best_iteration is not None else 999,
        "Validation_threshold": threshold,
        "Validation_F1": val_f1,
        "Test_Precision": precision_score(y_test, test_pred, zero_division=0),
        "Test_Recall": recall_score(y_test, test_pred, zero_division=0),
        "Test_F1": f1_score(y_test, test_pred, zero_division=0),
        "Test_PR_AUC": average_precision_score(y_test, test_prob),
        "Test_ROC_AUC": roc_auc_score(y_test, test_prob),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }

    # Save reproducibility outputs for this run.
    model.save_model(os.path.join(RESULTS_DIR, name.replace(" ", "_").lower() + ".json"))
    pd.DataFrame({
        "y_true": y_test.to_numpy(),
        "fraud_probability": test_prob,
        "prediction": test_pred
    }).to_csv(os.path.join(RESULTS_DIR, name.replace(" ", "_").lower() + "_predictions.csv"), index=False)

    del X_train, X_val, X_test, model
    gc.collect()
    return result

experiments = [
    ("Full-feature baseline", False, False),
    ("Without engineered features", True, False),
    ("Without identity features", False, True)
]

results = []
for name, no_eng, no_id in experiments:
    print("Running:", name)
    results.append(run_experiment(name, no_eng, no_id))
    display(pd.DataFrame(results).tail(1))


## 5. Compare results and calculate changes from the retrained baseline

In [ ]:
results_df = pd.DataFrame(results)

metric_cols = [
    "Test_Precision", "Test_Recall", "Test_F1",
    "Test_PR_AUC", "Test_ROC_AUC"
]

baseline = results_df.loc[
    results_df["Experiment"] == "Full-feature baseline", metric_cols
].iloc[0]

comparison = results_df.copy()
for metric in metric_cols:
    comparison[metric + "_Change_vs_Baseline"] = comparison[metric] - baseline[metric]

display(comparison.round(6))

results_df.to_csv(os.path.join(RESULTS_DIR, "ablation_results.csv"), index=False)
comparison.to_csv(os.path.join(RESULTS_DIR, "ablation_comparison_with_deltas.csv"), index=False)

print("Saved results to:", RESULTS_DIR)


## 6. Interpretation guide

- If removing engineered features lowers PR-AUC or F1, the engineered features contributed useful predictive information in this setup.
- If removing identity features lowers PR-AUC or F1, identity-related columns contributed useful information.
- If a metric increases after removal, the removed group may have added noise or encouraged overfitting in this split.
- Compare PR-AUC as a key metric because the fraud class is rare. Also discuss recall and precision together.
- These are feature-group ablations, not causal tests. They measure performance changes under this experimental setup.

**Reporting rule:** Use only the values generated in `ablation_results.csv`. Do not copy the old reported baseline metrics into the ablation results table: the baseline in this notebook is retrained under the same code path as both ablations.
